# Preprocesamiento y limpieza datasets

## 2. Datos semi-estructurados

Los datos semi-estructurados presentan una estructura parcial y jerárquica que
no encaja en el modelo tabular de un CSV directamente. A diferencia de los datos
tabulares (donde cada fila es un registro uniforme), aquí cada entidad puede
tener un número variable de sub-elementos con atributos distintos.

**Archivos a tratar:**
| Archivo | Formato | Desafío principal |
|---|---|---|
| `politicas_agricolas.xml` | XML jerárquico | Nodos hijo heterogéneos → 3 tablas |
| `condiciones_climaticas.json` | JSON anidado | Campo lista variable (eventos_extremos) |

**Herramientas utilizadas:**
- `xml.etree.ElementTree` — parser estándar de Python para XML
- `json` + `pandas.json_normalize` — aplanamiento de JSON anidado

### 2.1. políticas_agricolas.xml

**Problema técnico central — nodos heterogéneos:**
Los 3 tipos de nodo hijo tienen atributos distintos entre sí y además
atributos opcionales dentro del mismo tipo:
- `subsidio`: siempre tiene 6 atributos fijos
- `regulacion`: 3 atributos fijos + hasta 2 opcionales (multa_hectarea,
  multa_incumplimiento, porcentaje_minimo) según el tipo de regulación
- `acuerdo_internacional`: 2 atributos fijos + hasta 2 opcionales
  (anio_firma, objetivo_reduccion_emisiones, metas_agricolas_sostenibles)

**Estrategia elegida: 3 DataFrames independientes**
Aplanar los 3 tipos en una sola tabla generaría decenas de columnas vacías
y violaría los principios de normalización. La solución correcta es extraer
cada tipo de nodo en su propio DataFrame, todos enlazables por `codigo_pais`.
Esto permite joins flexibles en Fase 2 (EDA) y Fase 3 (ML) según necesidad.

#### Celda 1 - Carga de datos y exploración inicial

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd


ruta_xml = '../../data/raw/semi_structured/politicas_agricolas.xml'
tree = ET.parse(ruta_xml)
root = tree.getroot()

paises = root.findall('pais')

print(f"✅ Archivo XML cargado correctamente")
print(f"🌍 Número de países: {len(paises)}")
print(f"\n{'País':<20} {'Subsidios':>10} {'Regulaciones':>14} {'Acuerdos':>10} {'Total':>8}")
print("-" * 65)

total_subsidios = total_regulaciones = total_acuerdos = 0

for pais in paises:
    nombre = pais.get('nombre')
    n_sub  = len(pais.findall('subsidio'))
    n_reg  = len(pais.findall('regulacion'))
    n_ac   = len(pais.findall('acuerdo_internacional'))
    total  = n_sub + n_reg + n_ac
    total_subsidios    += n_sub
    total_regulaciones += n_reg
    total_acuerdos     += n_ac
    print(f"{nombre:<20} {n_sub:>10} {n_reg:>14} {n_ac:>10} {total:>8}")

print("-" * 65)
print(f"{'TOTAL':<20} {total_subsidios:>10} {total_regulaciones:>14} {total_acuerdos:>10} "
      f"{total_subsidios+total_regulaciones+total_acuerdos:>8}")

# Inventario de atributos opcionales en regulaciones
print(f"\n🔍 Atributos opcionales detectados en <regulacion>:")
atribs_reg = set()
for pais in paises:
    for reg in pais.findall('regulacion'):
        atribs_reg.update(reg.attrib.keys())
print(f"  {sorted(atribs_reg)}")

print(f"\n🔍 Atributos opcionales detectados en <acuerdo_internacional>:")
atribs_ac = set()
for pais in paises:
    for ac in pais.findall('acuerdo_internacional'):
        atribs_ac.update(ac.attrib.keys())
print(f"  {sorted(atribs_ac)}")

✅ Archivo XML cargado correctamente
🌍 Número de países: 12

País                  Subsidios   Regulaciones   Acuerdos    Total
-----------------------------------------------------------------
Argentina                     4              3          1        8
Brasil                        3              3          3        9
Estados Unidos                3              2          2        7
India                         3              2          2        7
China                         3              3          1        7
Francia                       4              2          3        9
Alemania                      3              2          3        8
Australia                     3              2          2        7
México                        3              2          2        7
Kenia                         2              3          1        6
Nueva Zelanda                 3              3          1        7
España                        3              2          1        6
---

#### Celda 2 - Parseo -> df_subsidios

In [2]:
registros_sub = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for sub in pais.findall('subsidio'):
        registros_sub.append({
            'codigo_pais'  : codigo,
            'nombre_pais'  : nombre,
            'tipo'         : sub.get('tipo'),
            'anio'         : sub.get('anio'),
            'monto'        : sub.get('monto'),
            'moneda'       : sub.get('moneda'),
            'beneficiarios': sub.get('beneficiarios'),
            'descripcion'  : sub.get('descripcion')
        })

df_subsidios = pd.DataFrame(registros_sub)
print(f"✅ df_subsidios: {df_subsidios.shape[0]} filas × {df_subsidios.shape[1]} columnas")
print(f"\n📋 Tipos de subsidio únicos ({df_subsidios['tipo'].nunique()}):")
print(df_subsidios['tipo'].value_counts().to_string())
df_subsidios.head(4)

✅ df_subsidios: 37 filas × 8 columnas

📋 Tipos de subsidio únicos (10):
tipo
investigacion_desarrollo       7
expansion_frontera_agricola    5
fertilizantes                  5
agricultura_organica           5
maquinaria                     4
bioenergia                     3
credito_rural                  2
seguro_agricola                2
conservacion_suelos            2
riego                          2


,codigo_pais,nombre_pais,tipo,anio,monto,moneda,beneficiarios,descripcion
0,ARG,Argentina,maquinaria,2021,25542386,USD,1914,Subsidio para maquinaria
1,ARG,Argentina,expansion_frontera_agricola,2023,12595066,USD,10898,Subsidio para expansion frontera agricola
2,ARG,Argentina,investigacion_desarrollo,2019,15562206,USD,7215,Subsidio para investigacion desarrollo
3,ARG,Argentina,fertilizantes,2021,32694726,USD,10823,Subsidio para fertilizantes


#### Celda 3 - parseo df_regulaciones

In [3]:
registros_reg = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for reg in pais.findall('regulacion'):
        registros_reg.append({
            'codigo_pais'          : codigo,
            'nombre_pais'          : nombre,
            'tipo'                 : reg.get('tipo'),
            'vigente'              : reg.get('vigente'),
            'anio_implementacion'  : reg.get('anio_implementacion'),
            'multa_hectarea'       : reg.get('multa_hectarea'),       # opcional
            'multa_incumplimiento' : reg.get('multa_incumplimiento'), # opcional
            'porcentaje_minimo'    : reg.get('porcentaje_minimo')     # opcional
        })

df_regulaciones = pd.DataFrame(registros_reg)
print(f"✅ df_regulaciones: {df_regulaciones.shape[0]} filas × {df_regulaciones.shape[1]} columnas")

print(f"\n🕳️  Nulos por columna (esperados en opcionales):")
print(df_regulaciones.isnull().sum().to_string())

print(f"\n📋 Tipos de regulación únicos ({df_regulaciones['tipo'].nunique()}):")
print(df_regulaciones['tipo'].value_counts().to_string())
df_regulaciones.head(4)

✅ df_regulaciones: 29 filas × 8 columnas

🕳️  Nulos por columna (esperados en opcionales):
codigo_pais              0
nombre_pais              0
tipo                     0
vigente                  0
anio_implementacion      0
multa_hectarea          26
multa_incumplimiento    25
porcentaje_minimo       24

📋 Tipos de regulación únicos (8):
tipo
pesticidas_neonicotinoides    6
reservas_legales              5
uso_agua                      4
etanol_mezcla                 4
rotacion_cultivos             4
deforestacion                 3
emisiones_ganaderas           2
bienestar_animal              1


,codigo_pais,nombre_pais,tipo,vigente,anio_implementacion,multa_hectarea,multa_incumplimiento,porcentaje_minimo
0,ARG,Argentina,pesticidas_neonicotinoides,true,2010,NaN,NaN,NaN
1,ARG,Argentina,uso_agua,false,2012,NaN,NaN,NaN
2,ARG,Argentina,etanol_mezcla,false,2013,NaN,NaN,NaN
3,BRA,Brasil,deforestacion,true,2018,4548,NaN,NaN


#### Celda 4 -> df_acuerdos

In [5]:
registros_ac = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for ac in pais.findall('acuerdo_internacional'):
        registros_ac.append({
            'codigo_pais'                   : codigo,
            'nombre_pais'                   : nombre,
            'nombre_acuerdo'                : ac.get('nombre'),
            'firmado'                       : ac.get('firmado'),
            'anio_firma'                    : ac.get('anio_firma'),
            'objetivo_reduccion_emisiones'  : ac.get('objetivo_reduccion_emisiones'),
            'metas_agricolas_sostenibles'   : ac.get('metas_agricolas_sostenibles')
        })

df_acuerdos = pd.DataFrame(registros_ac)
print(f"✅ df_acuerdos: {df_acuerdos.shape[0]} filas × {df_acuerdos.shape[1]} columnas")

print(f"\n🕳️  Nulos por columna (esperados en opcionales):")
print(df_acuerdos.isnull().sum().to_string())

print(f"\n📋 Acuerdos únicos ({df_acuerdos['nombre_acuerdo'].nunique()}):")
print(df_acuerdos['nombre_acuerdo'].value_counts().to_string())
df_acuerdos.head(4)

✅ df_acuerdos: 22 filas × 7 columnas

🕳️  Nulos por columna (esperados en opcionales):
codigo_pais                      0
nombre_pais                      0
nombre_acuerdo                   0
firmado                          0
anio_firma                       8
objetivo_reduccion_emisiones    21
metas_agricolas_sostenibles     21

📋 Acuerdos únicos (6):
nombre_acuerdo
Convenio_Biodiversidad    6
Acuerdo_Mercosur          5
Protocolo_Kyoto           4
TLCAN                     4
Acuerdo_Paris             2
ODS_2030                  1


,codigo_pais,nombre_pais,nombre_acuerdo,firmado,anio_firma,objetivo_reduccion_emisiones,metas_agricolas_sostenibles
0,ARG,Argentina,Convenio_Biodiversidad,false,NaN,NaN,NaN
1,BRA,Brasil,Protocolo_Kyoto,true,2009,NaN,NaN
2,BRA,Brasil,Acuerdo_Mercosur,true,2006,NaN,NaN
3,BRA,Brasil,Convenio_Biodiversidad,false,NaN,NaN,NaN


#### Celda 5 - Limpieza dataframes

In [6]:
# ── df_subsidios ──────────────────────────────────────────
df_subsidios['anio']          = df_subsidios['anio'].astype(int)
df_subsidios['monto']         = df_subsidios['monto'].astype(float)
df_subsidios['beneficiarios'] = df_subsidios['beneficiarios'].astype(int)
df_subsidios['tipo']          = df_subsidios['tipo'].str.strip()
df_subsidios['nombre_pais']   = df_subsidios['nombre_pais'].str.strip()

# ── df_regulaciones ───────────────────────────────────────
df_regulaciones['vigente']             = df_regulaciones['vigente'].map({'true': True, 'false': False})
df_regulaciones['anio_implementacion'] = df_regulaciones['anio_implementacion'].astype(int)
df_regulaciones['multa_hectarea']      = pd.to_numeric(df_regulaciones['multa_hectarea'], errors='coerce')
df_regulaciones['multa_incumplimiento']= pd.to_numeric(df_regulaciones['multa_incumplimiento'], errors='coerce')
df_regulaciones['porcentaje_minimo']   = pd.to_numeric(df_regulaciones['porcentaje_minimo'], errors='coerce')
df_regulaciones['tipo']                = df_regulaciones['tipo'].str.strip()

# ── df_acuerdos ───────────────────────────────────────────
df_acuerdos['firmado']                       = df_acuerdos['firmado'].map({'true': True, 'false': False})
df_acuerdos['anio_firma']                    = pd.to_numeric(df_acuerdos['anio_firma'], errors='coerce').astype('Int64')
df_acuerdos['objetivo_reduccion_emisiones']  = pd.to_numeric(df_acuerdos['objetivo_reduccion_emisiones'], errors='coerce')
df_acuerdos['metas_agricolas_sostenibles']   = pd.to_numeric(df_acuerdos['metas_agricolas_sostenibles'], errors='coerce')
df_acuerdos['nombre_acuerdo']                = df_acuerdos['nombre_acuerdo'].str.strip()

print("✅ Limpieza completada en los 3 DataFrames")
print(f"\ndf_subsidios    — tipos:\n{df_subsidios.dtypes}\n")
print(f"df_regulaciones — tipos:\n{df_regulaciones.dtypes}\n")
print(f"df_acuerdos     — tipos:\n{df_acuerdos.dtypes}")

✅ Limpieza completada en los 3 DataFrames

df_subsidios    — tipos:
codigo_pais          str
nombre_pais          str
tipo                 str
anio               int64
monto            float64
moneda               str
beneficiarios      int64
descripcion          str
dtype: object

df_regulaciones — tipos:
codigo_pais                 str
nombre_pais                 str
tipo                        str
vigente                  object
anio_implementacion       int64
multa_hectarea          float64
multa_incumplimiento    float64
porcentaje_minimo       float64
dtype: object

df_acuerdos     — tipos:
codigo_pais                         str
nombre_pais                         str
nombre_acuerdo                      str
firmado                            bool
anio_firma                        Int64
objetivo_reduccion_emisiones    float64
metas_agricolas_sostenibles     float64
dtype: object


#### Celda 6 - Validación

In [7]:
for nombre_df, df in [('df_subsidios', df_subsidios),
                       ('df_regulaciones', df_regulaciones),
                       ('df_acuerdos', df_acuerdos)]:
    print(f"{'='*50}")
    print(f"  {nombre_df}")
    print(f"{'='*50}")
    print(f"  Dimensiones : {df.shape}")
    print(f"  Nulos totales: {df.isnull().sum().sum()}")
    print(f"  Duplicados  : {df.duplicated().sum()}")
    print(f"  Países cubi.: {df['codigo_pais'].nunique()}\n")

  df_subsidios
  Dimensiones : (37, 8)
  Nulos totales: 0
  Duplicados  : 0
  Países cubi.: 12

  df_regulaciones
  Dimensiones : (29, 8)
  Nulos totales: 104
  Duplicados  : 0
  Países cubi.: 12

  df_acuerdos
  Dimensiones : (22, 7)
  Nulos totales: 50
  Duplicados  : 0
  Países cubi.: 12



**Nota sobre nulos en regulaciones y acuerdos:**
Los 104 nulos en `df_regulaciones` y 50 en `df_acuerdos` son **nulos estructurales**,
no errores de datos. Corresponden a atributos opcionales del XML que solo existen
para subconjuntos específicos de registros:
- `multa_hectarea` → solo en regulaciones de tipo `deforestacion`
- `multa_incumplimiento` → solo en regulaciones de tipo `rotacion_cultivos`
- `porcentaje_minimo` → solo en regulaciones de tipo `reservas_legales`
- `anio_firma` → solo cuando `firmado = True`

Estos nulos **no se imputan** — su ausencia es información en sí misma.
En ML se tratarán con un indicador binario o simplemente como 0 según el modelo.

#### Celda 7 - Exportación

In [8]:
import os
output_path = '../../data/processed/'
os.makedirs(output_path, exist_ok=True)

df_subsidios.to_csv(output_path + 'politicas_subsidios.csv',
                    index=False, encoding='utf-8-sig')
df_regulaciones.to_csv(output_path + 'politicas_regulaciones.csv',
                       index=False, encoding='utf-8-sig')
df_acuerdos.to_csv(output_path + 'politicas_acuerdos.csv',
                   index=False, encoding='utf-8-sig')

print("✅ Exportación completada:")
for archivo in ['politicas_subsidios.csv','politicas_regulaciones.csv','politicas_acuerdos.csv']:
    ruta = output_path + archivo
    print(f"   {archivo:<40} {os.path.getsize(ruta)/1024:.1f} KB")

✅ Exportación completada:
   politicas_subsidios.csv                  3.3 KB
   politicas_regulaciones.csv               1.3 KB
   politicas_acuerdos.csv                   1.0 KB


### 2.2. condiciones_climaticas.json

**Estructura jerárquica del archivo (3 niveles)**

**Desafío principal — campo `eventos_extremos`:**
Es una lista de longitud variable por registro: puede ser `[]` o contener
entre 1 y N strings (ej. `["ola_calor", "sequia_moderada"]`).
Esto impide una conversión directa a tabla plana.

**Estrategia de aplanamiento:**
1. `pd.json_normalize` para aplanar los 3 niveles de anidamiento
2. Campo `eventos_extremos` → 3 representaciones complementarias:
   - `eventos_str`: string concatenado con `|` (útil para filtros en EDA)
   - `n_eventos`: conteo numérico (útil como feature en ML)
   - One-hot encoding por tipo de evento → columna binaria por evento único

#### Celda 1 - Carga y exploración inicial

In [10]:
import json

ruta_json = '../../data/raw/semi_structured/condiciones_climaticas.json'

with open(ruta_json, 'r', encoding='utf-8') as f:
    datos_clima = json.load(f)

print(f"✅ JSON cargado correctamente")
print(f"🌍 Número de países: {len(datos_clima)}")

total_regiones = 0
total_registros = 0
eventos_unicos = set()

print(f"\n{'País':<22} {'Regiones':>10} {'Años/región':>12} {'Registros':>11}")
print("-" * 58)

for pais in datos_clima:
    nombre = pais['pais']
    n_reg  = len(pais['regiones'])
    anios_por_region = [len(r['datos_climaticos']) for r in pais['regiones']]
    n_registros = sum(anios_por_region)
    total_regiones  += n_reg
    total_registros += n_registros
    for region in pais['regiones']:
        for dato in region['datos_climaticos']:
            eventos_unicos.update(dato.get('eventos_extremos', []))
    print(f"{nombre:<22} {n_reg:>10} {str(set(anios_por_region)):>12} {n_registros:>11}")

print("-" * 58)
print(f"{'TOTAL':<22} {total_regiones:>10} {'':>12} {total_registros:>11}")

print(f"\n🌪️  Tipos de eventos extremos únicos ({len(eventos_unicos)}):")
for e in sorted(eventos_unicos):
    print(f"   - {e}")

✅ JSON cargado correctamente
🌍 Número de países: 10

País                     Regiones  Años/región   Registros
----------------------------------------------------------
Argentina                       3         {10}          30
Brasil                          3         {10}          30
Estados Unidos                  3         {10}          30
India                           3         {10}          30
China                           1         {10}          10
Francia                         1         {10}          10
Alemania                        1         {10}          10
Australia                       3         {10}          30
México                          1         {10}          10
Kenia                           1         {10}          10
----------------------------------------------------------
TOTAL                          20                      200

🌪️  Tipos de eventos extremos únicos (9):
   - granizo
   - helada_tardia
   - incendios
   - inundacion
   - ola_calor


**Resultado de la exploración:**
- 10 países, 20 regiones, 200 registros (10 años × región, completamente uniforme)
- 9 tipos de eventos extremos únicos → 9 columnas one-hot
- Estructura perfectamente regular: todos los países tienen exactamente 10
  años por región → 0 irregularidades a gestionar

**Estrategia de aplanamiento — 2 pasos:**

**Paso 1 — `json_normalize` manual iterando sobre los 3 niveles:**
`pd.json_normalize` con `record_path` no gestiona bien 3 niveles de anidamiento
con metadatos en cada nivel. La estrategia más robusta y legible es un bucle
explícito que construye registros planos directamente, preservando `pais`,
`codigo_iso` y `region` como metadatos en cada fila.

**Paso 2 — Tratamiento de `eventos_extremos` (lista variable):**
Se generan 3 representaciones complementarias para distintos usos:

| Representación | Ejemplo | Uso |
|---|---|---|
| `eventos_str` | `"ola_calor\|sequia_moderada"` | Filtros y visualización en EDA |
| `n_eventos` | `2` | Feature numérica directa en ML |
| One-hot (`ev_ola_calor`, etc.) | `1` / `0` | Features binarias para modelos |

Los 9 eventos únicos detectados generan 9 columnas con prefijo `ev_`.

#### Celda 2 - Aplanamiento

In [11]:
registros_clima = []

for pais in datos_clima:
    nombre_pais = pais['pais']
    codigo_iso  = pais['codigo_iso']
    for region in pais['regiones']:
        nombre_region = region['nombre']
        for dato in region['datos_climaticos']:
            fila = {
                'pais'                     : nombre_pais,
                'codigo_iso'               : codigo_iso,
                'region'                   : nombre_region,
                'anio'                     : dato['anio'],
                'temperatura_promedio'     : dato['temperatura_promedio'],
                'temperatura_maxima'       : dato['temperatura_maxima'],
                'temperatura_minima'       : dato['temperatura_minima'],
                'precipitacion_total'      : dato['precipitacion_total'],
                'humedad_relativa_promedio': dato['humedad_relativa_promedio'],
                'dias_con_heladas'         : dato['dias_con_heladas'],
                'meses_estres_hidrico'     : dato['meses_estres_hidrico'],
                'indice_aridez'            : dato['indice_aridez'],
                'eventos_extremos'         : dato.get('eventos_extremos', [])
            }
            registros_clima.append(fila)

df_clima = pd.DataFrame(registros_clima)

print(f"✅ DataFrame base construido")
print(f"📐 Dimensiones: {df_clima.shape}")
print(f"🕳️  Nulos: {df_clima.isnull().sum().sum()}")
print(f"\n🔍 Muestra del campo eventos_extremos (raw):")
print(df_clima[['pais','region','anio','eventos_extremos']].head(8).to_string())

✅ DataFrame base construido
📐 Dimensiones: (200, 13)
🕳️  Nulos: 0

🔍 Muestra del campo eventos_extremos (raw):
        pais        region  anio              eventos_extremos
0  Argentina  Pampa Húmeda  2011                            []
1  Argentina  Pampa Húmeda  2012                            []
2  Argentina  Pampa Húmeda  2013  [ola_calor, sequia_moderada]
3  Argentina  Pampa Húmeda  2014                            []
4  Argentina  Pampa Húmeda  2015                            []
5  Argentina  Pampa Húmeda  2016                            []
6  Argentina  Pampa Húmeda  2017                            []
7  Argentina  Pampa Húmeda  2018                            []


#### Celda 3 - Tratamiento de eventos extremos

In [12]:
EVENTOS_UNICOS = sorted([
    'granizo', 'helada_tardia', 'incendios', 'inundacion',
    'ola_calor', 'sequia_extrema', 'sequia_moderada',
    'sequia_severa', 'tormenta_severa'
])

# 1. eventos_str: lista → string separado por '|' (vacío → 'ninguno')
df_clima['eventos_str'] = df_clima['eventos_extremos'].apply(
    lambda lst: '|'.join(lst) if lst else 'ninguno'
)

# 2. n_eventos: conteo por registro
df_clima['n_eventos'] = df_clima['eventos_extremos'].apply(len)

# 3. One-hot: una columna binaria por evento
for evento in EVENTOS_UNICOS:
    col = f"ev_{evento}"
    df_clima[col] = df_clima['eventos_extremos'].apply(
        lambda lst: 1 if evento in lst else 0
    )

# Eliminar columna raw
df_clima = df_clima.drop(columns=['eventos_extremos'])

print(f"✅ Campo eventos_extremos procesado")
print(f"📐 Dimensiones tras aplanamiento: {df_clima.shape}")
print(f"\n📊 Frecuencia de cada evento extremo:")
for evento in EVENTOS_UNICOS:
    col = f"ev_{evento}"
    n   = df_clima[col].sum()
    print(f"   {evento:<25}: {n:>3} registros ({n/len(df_clima)*100:.1f}%)")

print(f"\n📊 Distribución de n_eventos por registro:")
print(df_clima['n_eventos'].value_counts().sort_index().to_string())

✅ Campo eventos_extremos procesado
📐 Dimensiones tras aplanamiento: (200, 23)

📊 Frecuencia de cada evento extremo:
   granizo                  :   8 registros (4.0%)
   helada_tardia            :  16 registros (8.0%)
   incendios                :  11 registros (5.5%)
   inundacion               :  11 registros (5.5%)
   ola_calor                :  12 registros (6.0%)
   sequia_extrema           :  17 registros (8.5%)
   sequia_moderada          :  11 registros (5.5%)
   sequia_severa            :  19 registros (9.5%)
   tormenta_severa          :  13 registros (6.5%)

📊 Distribución de n_eventos por registro:
n_eventos
0    142
1     20
2     16
3     22


#### Celda 4 - Limpieza y conversión de tipos

In [13]:
# Tipos numéricos (ya deberían ser correctos, forzamos por robustez)
cols_float = [
    'temperatura_promedio', 'temperatura_maxima', 'temperatura_minima',
    'precipitacion_total', 'humedad_relativa_promedio', 'dias_con_heladas',
    'meses_estres_hidrico', 'indice_aridez'
]
for col in cols_float:
    df_clima[col] = pd.to_numeric(df_clima[col], errors='coerce')

df_clima['anio'] = df_clima['anio'].astype(int)

# Normalizar strings
df_clima['pais']       = df_clima['pais'].str.strip()
df_clima['region']     = df_clima['region'].str.strip()
df_clima['codigo_iso'] = df_clima['codigo_iso'].str.strip()

print("✅ Tipos convertidos correctamente")
print(f"\n📋 Tipos de datos finales:")
print(df_clima.dtypes)
print(f"\n🕳️  Nulos post-limpieza: {df_clima.isnull().sum().sum()}")

✅ Tipos convertidos correctamente

📋 Tipos de datos finales:
pais                             str
codigo_iso                       str
region                           str
anio                           int64
temperatura_promedio         float64
temperatura_maxima           float64
temperatura_minima           float64
precipitacion_total            int64
humedad_relativa_promedio      int64
dias_con_heladas               int64
meses_estres_hidrico           int64
indice_aridez                float64
eventos_str                      str
n_eventos                      int64
ev_granizo                     int64
ev_helada_tardia               int64
ev_incendios                   int64
ev_inundacion                  int64
ev_ola_calor                   int64
ev_sequia_extrema              int64
ev_sequia_moderada             int64
ev_sequia_severa               int64
ev_tormenta_severa             int64
dtype: object

🕳️  Nulos post-limpieza: 0


#### Celda 5 - Validación

In [14]:
print("=" * 55)
print("  VALIDACIÓN FINAL — condiciones_climaticas_processed")
print("=" * 55)

print(f"\n📐 Dimensiones finales  : {df_clima.shape}")
print(f"🕳️  Nulos totales        : {df_clima.isnull().sum().sum()}")
print(f"🔁 Duplicados           : {df_clima.duplicated().sum()}")
print(f"\n📅 Rango temporal: {df_clima['anio'].min()} → {df_clima['anio'].max()}")
print(f"🌍 Países         : {df_clima['pais'].nunique()} ({sorted(df_clima['pais'].unique())})")
print(f"🗺️  Regiones       : {df_clima['region'].nunique()}")

print(f"\n📊 Estadísticas climáticas clave:")
print(df_clima[['temperatura_promedio','precipitacion_total',
                'indice_aridez','meses_estres_hidrico',
                'n_eventos']].describe().round(2).to_string())

print(f"\n📊 Registros con al menos 1 evento extremo: "
      f"{(df_clima['n_eventos'] > 0).sum()} "
      f"({(df_clima['n_eventos'] > 0).mean()*100:.1f}%)")

print(f"\n🔍 Primeras 3 filas:")
df_clima[['pais','region','anio','temperatura_promedio',
          'precipitacion_total','eventos_str','n_eventos']].head(3)

  VALIDACIÓN FINAL — condiciones_climaticas_processed

📐 Dimensiones finales  : (200, 23)
🕳️  Nulos totales        : 0
🔁 Duplicados           : 0

📅 Rango temporal: 2011 → 2020
🌍 Países         : 10 (['Alemania', 'Argentina', 'Australia', 'Brasil', 'China', 'Estados Unidos', 'Francia', 'India', 'Kenia', 'México'])
🗺️  Regiones       : 16

📊 Estadísticas climáticas clave:
       temperatura_promedio  precipitacion_total  indice_aridez  meses_estres_hidrico  n_eventos
count                200.00               200.00         200.00                200.00     200.00
mean                  17.78               902.60          15.04                  3.80       0.59
std                    4.31               349.60           7.16                  2.97       1.03
min                    8.70               281.00           3.73                  0.00       0.00
25%                   13.98               609.50           9.32                  1.00       0.00
50%                   18.25               89

,pais,region,anio,temperatura_promedio,precipitacion_total,eventos_str,n_eventos
0,Argentina,Pampa Húmeda,2011,16.9,1050,ninguno,0
1,Argentina,Pampa Húmeda,2012,18.7,518,ninguno,0
2,Argentina,Pampa Húmeda,2013,21.2,1390,ola_calor|sequia_moderada,2


Sin nulos, sin duplicados y datos con rangos realistas para el conjunto de paises.

#### Celda 6 - Exportación

In [15]:
archivo_clima = output_path + 'condiciones_climaticas_processed.csv'
df_clima.to_csv(archivo_clima, index=False, encoding='utf-8-sig')

print(f"✅ Exportación completada:")
print(f"   {'condiciones_climaticas_processed.csv':<45} "
      f"{os.path.getsize(archivo_clima)/1024:.1f} KB")
print(f"\n📋 Columnas exportadas ({len(df_clima.columns)}):")
for i, col in enumerate(df_clima.columns, 1):
    print(f"   {i:02d}. {col}")

✅ Exportación completada:
   condiciones_climaticas_processed.csv          19.4 KB

📋 Columnas exportadas (23):
   01. pais
   02. codigo_iso
   03. region
   04. anio
   05. temperatura_promedio
   06. temperatura_maxima
   07. temperatura_minima
   08. precipitacion_total
   09. humedad_relativa_promedio
   10. dias_con_heladas
   11. meses_estres_hidrico
   12. indice_aridez
   13. eventos_str
   14. n_eventos
   15. ev_granizo
   16. ev_helada_tardia
   17. ev_incendios
   18. ev_inundacion
   19. ev_ola_calor
   20. ev_sequia_extrema
   21. ev_sequia_moderada
   22. ev_sequia_severa
   23. ev_tormenta_severa
